Planar robots with rotary joints
================================

**Topic**: Path planning for planar robots with rotary joints

Version | Author | Remark
------------ | ------------- | -------------
0.1 | Gergely Sotí | Base concept implemented and prepared
0.2 | Björn Hein | Minor textual additions and corrections
0.3 | Björn Hein | Reworked: kinematics, collision checker from the module, C-space, resolution of the line check, more joints

This worksheet **explains how the planners of the course are applied to a robot with joints**: the kinematics of a planar robot, its collision checker, what a line in the joint space means in the work space, and planning for robots with two and more joints.

License is based on Creative Commons: Attribution-NonCommercial 4.0 International (CC BY-NC 4.0) (please check: http://creativecommons.org/licenses/by-nc/4.0/)

**Prerequisite**:

* IP-2-0-Pathplanning-Introduction (configuration space, implicit C-space), IP-4-0-Test-Environments (collision checker) and the planners: IP-5 (Basic PRM), IP-6-0 (Visibility-PRM), IP-7-0 (Lazy-PRM)

**Dependencies**:

* IPPlanarManipulator.py (`PlanarJoint`, `PlanarRobot`; needs the package `sympy`), IPEnvironmentKin.py (`KinChainCollisionChecker`, `animateSolution`), IPVISPlanarManipulator.py (figures of this notebook), IPPerfMonitor.py and the planners with their visualizations (IPBasicPRM.py, IPVisibilityPRM.py, IPLazyPRM.py, IPVIS*.py)

Important links are:

* General Info: https://www.python.org
* SymPy: https://www.sympy.org
* NetworkX: https://networkx.org
* Shapely: https://shapely.readthedocs.io/
* Matplotlib: https://matplotlib.org/stable/

Remember that:

* you have to press ctrl-return or shift-return, to execute the code in the code sections, only then the variables are "generated" and can be used
* you can execute the whole notebook by "Run All"

# Goal

* Understanding the forward kinematics of a planar robot as a chain of homogeneous transformations
* Understanding how a collision checker for a robot works: configuration $\rightarrow$ forward kinematics $\rightarrow$ collision check in the work space
* Seeing that the planners are used **unchanged**: only the collision checker is exchanged
* Understanding what a straight line in the joint space means in the work space - and why the resolution of the line check matters
* Experiencing how the planners behave with more joints

# Idea

So far the robot was a point: its configuration $q = (x, y)$ was its position, and the obstacles of the scene were the C-obstacles. For a robot with joints (see IP-2-0):

* the **configuration** is the vector of the joint angles $q = (q_1, \dots, q_n)$, the C-space is the **joint space**,
* the obstacles are given in the **work space**; the C-obstacles are unknown,
* a configuration is checked by placing the robot in the work space: **forward kinematics**, then a geometric test.

The planners only use the interface of the collision checker (`getDim`, `getEnvironmentLimits`, `pointInCollision`, `lineInCollision`). Therefore only a new collision checker is needed: `KinChainCollisionChecker`.

In [ ]:
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp
from shapely.geometry import LineString

from IPPlanarManipulator import PlanarJoint, PlanarRobot
from IPEnvironmentKin import KinChainCollisionChecker
from IPPerfMonitor import IPPerfMonitor
from IPVISStyle import START_COLOR, GOAL_COLOR
import IPVISPlanarManipulator as vis

# Kinematics of a planar robot

## One joint

A link of the robot is described by the simplified Denavit-Hartenberg parameters of the planar case: the joint angle $\theta$ (the variable) and the length $a$ of the link (the parameters $d$ and $\alpha$ are omitted). The pose of the end of the link relative to its joint is a rotation by $\theta$ followed by a translation by $a$ along the rotated x-axis, written as homogeneous $3 \times 3$ matrix.

`PlanarJoint` (IPPlanarManipulator.py) builds this matrix **symbolically** with SymPy:

In [ ]:
joint = PlanarJoint()      # default: a = 1.5, theta = 0
joint.M

The last column contains the position of the end of the link. For numerical values the symbolic expression is translated once into a NumPy function (`sympy.lambdify`); `get_transform()` evaluates it for the current angle:

In [ ]:
for theta in [0.0, 0.8, math.pi / 2]:
    joint.move(theta)
    print(f"theta = {theta:.2f}: end of the link at {joint.get_transform()}")

## A chain of joints

`PlanarRobot` multiplies the matrices of its joints: $T_k = M_1 \cdot M_2 \cdots M_k$ is the pose of the end of link $k$ in the base frame. The base of the robot is the origin. For two joints the position of the tip (tool center point, TCP) is the well-known formula:

In [ ]:
robot = PlanarRobot(n_joints=2)      # two links, each 1.5 long

tcp = sp.simplify(robot.Ms[2] * sp.Matrix([0, 0, 1]))
tcp[:2, :]

`move(q)` sets the joint angles, `get_transforms()` returns the positions of all joints in the work space: base, end of link 1, end of link 2 (TCP). All positions are computed by **one** generated function (`lambdify` with `cse=True`: common subexpressions like $\cos(\theta_0)$ are evaluated only once).

In [ ]:
robot.move([0.0, math.pi / 2])
for name, position in zip(["base", "joint 2", "TCP"], robot.get_transforms()):
    print(f"{name:8s} {np.round(position, 3)}")

## Forward and inverse kinematics

The forward kinematics is a function: one configuration gives one pose of the robot. The inverse direction is not unique: the two-link robot reaches (almost) every TCP position with **two** configurations, "elbow left" and "elbow right". For the planner these are two different points in the joint space - a goal given in the work space may correspond to several goal configurations (this is why `planPath` takes a *list* of goals).

In [ ]:
def inverseKinematics(x, y, a1=1.5, a2=1.5):
    "Both configurations of the two-link robot with the TCP at (x, y)"
    c2 = (x * x + y * y - a1 * a1 - a2 * a2) / (2 * a1 * a2)
    solutions = []
    for q2 in (math.acos(c2), -math.acos(c2)):
        q1 = math.atan2(y, x) - math.atan2(a2 * math.sin(q2), a1 + a2 * math.cos(q2))
        solutions.append([q1, q2])
    return solutions


jointLimits = [[-math.pi, math.pi], [-math.pi, math.pi]]
freeSpace = KinChainCollisionChecker(robot, dict(), limits=jointLimits)     # no obstacles

elbowConfigurations = inverseKinematics(1.5, 1.8)
for q in elbowConfigurations:
    robot.move(q)
    print("q =", np.round(q, 3), "-> TCP", np.round(robot.get_transforms()[-1], 3))

vis.showWorkAndCSpace(freeSpace, elbowConfigurations, labels=["A", "B"], colors=[vis.PATH_COLOR, vis.COMPONENT_COLOR])

# The collision checker

`KinChainCollisionChecker` (IPEnvironmentKin.py) is a subclass of `CollisionChecker`. The scene is the same kind of dict as before, but the obstacles now live in the **work space**. The essential methods (shortened):

```python
def robotInCollision(self, pos):
    self.kin_chain.move(pos)                              # 1. set the joint angles
    joint_positions = self.kin_chain.get_transforms()     # 2. forward kinematics
    for i in range(1, len(joint_positions)):              # 3. check every link in the work space
        if self.segmentInCollision(joint_positions[i-1], joint_positions[i]):
            return True
    return False

@IPPerfMonitor
def pointInCollision(self, pos):
    return self.robotInCollision(pos)

def segmentInCollision(self, startPos, endPos, radius=None):
    radius = self.link_radius if radius is None else radius
    link = LineString([(startPos[0], startPos[1]), (endPos[0], endPos[1])])
    for key, value in self.scene.items():
        if radius > 0.0:
            if value.distance(link) <= radius:            # link as capsule
                return True
        elif value.intersects(link):                      # link as line
            return True
    return False

@IPPerfMonitor
def lineInCollision(self, startPos, endPos, steps=None):
    for pos in interpolate_line(startPos, endPos, self.fk_resolution):
        if self.pointInCollision(pos):
            return True
    return False
```

* A **"point"** of the planner is a configuration: a complete pose of the robot.
* A link is a **line** between two joints (`link_radius=0`, default) or a **capsule** with radius `link_radius` around this line.
* A **"line"** of the planner is a straight line in the joint space: it is checked at configurations with the distance `fk_resolution` (see below).
* Not checked: the joint limits (they are only used by the planners for sampling) and collisions of the robot with itself.

The scene of this notebook (also used in IP-2-0, IP-10-1 and IP-12-0) with start and goal:

In [ ]:
obstacles = dict()
obstacles["obs1"] = LineString([(-2, 0), (-0.8, 0)]).buffer(0.5)
obstacles["obs2"] = LineString([(2, 0), (2, 1)]).buffer(0.2)
obstacles["obs3"] = LineString([(-1, 2), (1, 2)]).buffer(0.1)

environment = KinChainCollisionChecker(robot, obstacles, limits=jointLimits, fk_resolution=0.2)

start = [2.0, 0.5]
goal = [-2.0, -0.5]
blocked = [0.6, 1.2]
for name, q in [("start", start), ("goal", goal), ("blocked", blocked)]:
    print(f"{name:8s} q = {q}: in collision: {environment.pointInCollision(q)}")

vis.showConfigurations(environment, [start, goal, blocked], titles=["start", "goal", "blocked: in collision"],
                       colors=[START_COLOR, GOAL_COLOR, vis.PATH_COLOR])

## The joint space

For two joints the C-obstacles can be made visible by brute force (as in IP-2-0): `computeCSpace()` checks a grid of configurations. **This is only done for the figures** - the planners never see this picture, they only ask the collision checker for single configurations and lines. After `computeCSpace()` the method `drawObstacles(ax)` draws the sampled C-obstacles, so the visualizations of the planners show them.

Every robot pose on the left is one point on the right. The three small obstacles of the work space become large C-obstacles with completely different shapes.

In [ ]:
startTime = time.perf_counter()
grid = environment.computeCSpace(resolution=150)
duration = time.perf_counter() - startTime
print(f"{grid.size} collision checks in {duration:.2f} s, {grid.mean():.0%} of the configurations are in collision")

vis.showWorkAndCSpace(environment, [start, goal, blocked], labels=["start", "goal", "blocked"],
                      colors=[START_COLOR, GOAL_COLOR, vis.PATH_COLOR])

**Remark:** joint angles are periodic: $-\pi$ and $\pi$ are the same angle, strictly speaking this C-space is a torus (see IP-2-0). The collision checker and the planners of the course do not use this: the robot cannot move "over the border" of the limits.

## Links with a thickness

With `link_radius` the links are capsules. The robot gets "fatter" - and so do the C-obstacles. The configuration below is free for the line model and in collision for links with radius 0.15:

In [ ]:
thickEnvironment = KinChainCollisionChecker(robot, obstacles, limits=jointLimits, fk_resolution=0.2, link_radius=0.15)
thickGrid = thickEnvironment.computeCSpace(resolution=150)
print(f"in collision: links as lines {grid.mean():.0%}, links as capsules {thickGrid.mean():.0%} of the configurations")

close = [1.0, 0.0]
print("in collision: links as lines:", environment.pointInCollision(close), "- links as capsules:", thickEnvironment.pointInCollision(close))
vis.showWorkAndCSpace(environment, [close], labels=["C"], titles=("links as lines", "joint space"))
vis.showWorkAndCSpace(thickEnvironment, [close], labels=["C"], titles=("links as capsules, radius 0.15", "joint space"))

# Lines in the joint space

The planners connect two configurations by a **straight line in the joint space**: all joints move simultaneously and proportionally (a synchronous point-to-point motion). In the work space this is not a straight line: the tip moves on a curve.

`lineInCollision` checks this motion at configurations with the distance `fk_resolution` in the joint space. Between two checked configurations the robot moves: the tip of the two-link robot by up to $(a_1 + a_2) \, |\Delta q_1| + a_2 \, |\Delta q_2|$. With `fk_resolution = 0.2` this is up to about 0.6 - but the thinnest obstacle of the scene is only 0.2 thick.

An example: the same line, checked with the resolution 0.2 and with 0.01 (blue: free, orange: in collision).

In [ ]:
qa, qb = [-1.87, 0.29], [-0.46, 2.7]
print("lineInCollision with fk_resolution = 0.2:", environment.lineInCollision(qa, qb))

vis.showLineCheck(environment, qa, qb, resolutions=(0.2, 0.01))

With the resolution 0.2 the line is reported as free: the collision with the obstacle on the right happens **between** two checked configurations. The fine check finds it.

* A sampled line check can miss collisions (as in IP-4-0) - for a robot there is no exact alternative, as the C-obstacles are unknown.
* The resolution has to fit the robot and the scene: the motion of the robot between two checks should be smaller than the smallest obstacle. The price is the number of collision checks.

The following function checks a path independently of the planner with a fine resolution:

In [ ]:
def pathInCollision(collisionChecker, path, resolution=0.01):
    "Independent check of a path (list of configurations): True, if a configuration on it is in collision"
    return any(collisionChecker.robotInCollision(q)
               for a, b in zip(path[:-1], path[1:]) for q in vis.sampleLine(a, b, resolution))


def pathLength(path):
    "Length of a path in the joint space [rad]"
    return sum(math.dist(a, b) for a, b in zip(path[:-1], path[1:]))

# Path planning

The planners are used exactly as for the point robot: they get the collision checker, start and goal are configurations. `runPlanner` fixes the seed, runs a planner and collects some measures (number of collision checks from `IPPerfMonitor`, see IP-X-0); the path is checked with `pathInCollision`.

In [ ]:
from IPBasicPRM import BasicPRM
from IPVISBasicPRM import basicPRMVisualize
from IPVisibilityPRM import VisPRM
from IPVISVisibilityPRM import visibilityPRMVisualize
from IPLazyPRM import LazyPRM
from IPVISLazyPRM import lazyPRMVisualize


def runPlanner(plannerClass, config, collisionChecker, start, goal, seed):
    "One run of a planner. Returns (planner, solution, measures)"
    random.seed(seed)
    np.random.seed(seed)
    IPPerfMonitor.clearData()
    planner = plannerClass(collisionChecker)
    startTime = time.perf_counter()
    solution = planner.planPath([start], [goal], config) or []
    duration = time.perf_counter() - startTime
    calls = IPPerfMonitor.dataFrame()
    path = vis.solutionConfigurations(planner, solution)
    measures = {"path found": len(solution) > 0,
                "path valid": len(solution) > 0 and not pathInCollision(collisionChecker, path),
                "point checks": int((calls.name == "pointInCollision").sum()),
                "time [s]": round(duration, 3),
                "nodes in path": len(solution),
                "path length [rad]": round(pathLength(path), 2) if solution else float("nan")}
    return planner, solution, measures


planners = dict()
planners["Basic PRM"] = (BasicPRM, {"radius": 1.5, "numNodes": 150}, basicPRMVisualize)
planners["Visibility-PRM"] = (VisPRM, {"ntry": 50}, visibilityPRMVisualize)
planners["Lazy-PRM"] = (LazyPRM, {"initialRoadmapSize": 40, "updateRoadmapSize": 20, "kNearest": 8,
                                  "maxIterations": 40}, lazyPRMVisualize)

One run of every planner. Left the roadmap in the joint space (drawn by the usual visualization of the planner, the C-obstacles in the background are only for our orientation), right the resulting motion of the robot: the later the pose, the more opaque; blue the path of the tip.

In [ ]:
rows = []
for name, (plannerClass, config, visualizer) in planners.items():
    planner, solution, measures = runPlanner(plannerClass, config, environment, start, goal, seed=1)
    rows.append({"planner": name, **measures})
    vis.showSolution(environment, planner, solution, visualizer, title=name)

pd.DataFrame(rows).set_index("planner")

Observations:

* All three planners solve the problem without any change - they do not know that the "point" they move is a robot with two joints.
* The motions are valid, but not nice: the robot swings around, because the paths follow random nodes. Smoothing (IP-12-0) improves this.
* The Lazy-PRM needs by far the fewest collision checks: most of this C-space is free.

## Is the path really collision-free?

The planners trust `lineInCollision`. With `fk_resolution = 0.2` this check can miss collisions (see above). Ten runs per planner, once with the resolution 0.2 and once with 0.02; every path is checked independently with the resolution 0.01:

In [ ]:
fineEnvironment = KinChainCollisionChecker(robot, obstacles, limits=jointLimits, fk_resolution=0.02)

rows = []
for resolution, collisionChecker in [(0.2, environment), (0.02, fineEnvironment)]:
    for name, (plannerClass, config, visualizer) in planners.items():
        runs = pd.DataFrame([runPlanner(plannerClass, config, collisionChecker, start, goal, seed)[2]
                             for seed in range(10)])
        rows.append({"fk_resolution": resolution, "planner": name,
                     "paths found": int(runs["path found"].sum()),
                     "paths valid": int(runs["path valid"].sum()),
                     "point checks (mean)": int(runs["point checks"].mean()),
                     "time [s] (mean)": round(runs["time [s]"].mean(), 3)})

validation = pd.DataFrame(rows).set_index(["fk_resolution", "planner"])
validation

Observations:

* With `fk_resolution = 0.2` every planner finds a path in all runs - but one path of the Basic PRM and one of the Lazy-PRM lead **through an obstacle**. Nothing in the result of the planner shows this.
* With `fk_resolution = 0.02` all paths are valid. The price: about five to eight times as many collision checks.
* A result should be verified independently of the planner (compare `validatePath` in the lab IP-7-1).

One of the invalid paths: the edge of the solution, which is in collision, with both resolutions.

In [ ]:
invalid = None
for seed in range(10):
    planner, solution, measures = runPlanner(LazyPRM, planners["Lazy-PRM"][1], environment, start, goal, seed)
    if measures["path found"] and not measures["path valid"]:
        invalid = (planner, solution, seed)
        break

if invalid:
    planner, solution, seed = invalid
    path = vis.solutionConfigurations(planner, solution)
    for a, b in zip(path[:-1], path[1:]):
        if pathInCollision(environment, [a, b]):
            print(f"seed {seed}: the edge {np.round(a, 2)} -> {np.round(b, 2)} of the solution is in collision")
            vis.showLineCheck(environment, a, b, resolutions=(0.2, 0.01))
            break
else:
    print("No invalid path among these runs.")

## Animation

`animateSolution` (IPEnvironmentKin.py) moves the robot along a solution path. For two joints it shows the joint space next to the work space. (Optional, takes some seconds; the robot is drawn as lines.)

In [ ]:
from IPEnvironmentKin import animateSolution

planner, solution, measures = runPlanner(LazyPRM, planners["Lazy-PRM"][1], fineEnvironment, start, goal, seed=1)
animateSolution(planner, fineEnvironment, solution, lazyPRMVisualize, workSpaceLimits=[[-3.3, 3.3], [-3.3, 3.3]])

# More joints

Nothing has to be changed for a robot with more joints: `PlanarRobot(n_joints=n)` builds the kinematics, the collision checker needs the limits of all joints. The dimension of the C-space is the number of joints.

Example: three joints, links with radius 0.05. The C-space is 3-dimensional (see IP-2-0 for a picture of such a C-space) - here only the work space is shown.

In [ ]:
robot3 = PlanarRobot(n_joints=3)
for joint in robot3.joints:
    joint.a = 1.0                    # length of every link: the same reach as the two-link robot

environment3 = KinChainCollisionChecker(robot3, obstacles, limits=[[-math.pi, math.pi]] * 3, fk_resolution=0.05,
                                        link_radius=0.05)
start3 = [2.0, 0.25, 0.25]
goal3 = [-2.0, -0.25, -0.25]

planner3, solution3, measures3 = runPlanner(LazyPRM, planners["Lazy-PRM"][1], environment3, start3, goal3, seed=1)
print(measures3)

path3 = vis.solutionConfigurations(planner3, solution3)
vis.showMotion(environment3, path3, "Lazy-PRM, three joints")
vis.showJointAngles(path3)

## How do the planners scale?

Robots with 2 to 8 joints and the same total length (3.0), comparable start and goal (first joint 2.0 resp. -2.0, the others slightly bent). The parameters of the planners stay the same - only the connection radius of the Basic PRM grows with $\sqrt{n}$, as distances in the joint space grow with the dimension. Five runs each (takes some seconds):

In [ ]:
def planarProblem(n):
    "Robot with n joints and total length 3, collision checker, start and goal"
    robotN = PlanarRobot(n_joints=n)
    for joint in robotN.joints:
        joint.a = 3.0 / n
    collisionChecker = KinChainCollisionChecker(robotN, obstacles, limits=[[-math.pi, math.pi]] * n,
                                                fk_resolution=0.05, link_radius=0.05)
    return collisionChecker, [2.0] + [0.5 / (n - 1)] * (n - 1), [-2.0] + [-0.5 / (n - 1)] * (n - 1)


rows = []
problems = dict()
for n in [2, 3, 4, 6, 8]:
    problems[n] = planarProblem(n)
    collisionChecker, startN, goalN = problems[n]
    configurations = {"Basic PRM": (BasicPRM, {"radius": 1.5 * math.sqrt(n / 2), "numNodes": 150}),
                      "Lazy-PRM": (LazyPRM, planners["Lazy-PRM"][1])}
    for name, (plannerClass, config) in configurations.items():
        runs = pd.DataFrame([runPlanner(plannerClass, config, collisionChecker, startN, goalN, seed)[2]
                             for seed in range(5)])
        rows.append({"joints": n, "planner": name, "paths found": int(runs["path found"].sum()),
                     "paths valid": int(runs["path valid"].sum()),
                     "point checks (mean)": int(runs["point checks"].mean()),
                     "time [s] (mean)": round(runs["time [s]"].mean(), 2),
                     "path length [rad] (mean)": round(runs["path length [rad]"].mean(), 1)})

pd.DataFrame(rows).set_index(["joints", "planner"])

Observations:

* The **Lazy-PRM** solves all problems with about the same number of collision checks: it only checks along candidate paths, and most of these C-spaces are free. The time grows slowly - one collision check tests more links.
* The **Basic PRM** with 150 nodes works for two and three joints and then fails more and more often: the volume of the C-space grows exponentially with the number of joints, 150 random nodes with a limited connection radius do not form a connected roadmap any more. It would need many more nodes - and checks every edge.
* The paths get longer with the number of joints: every random node moves **all** joints.

The motion of the robot with eight joints:

In [ ]:
collisionChecker8, start8, goal8 = problems[8]
planner8, solution8, measures8 = runPlanner(LazyPRM, planners["Lazy-PRM"][1], collisionChecker8, start8, goal8, seed=1)
path8 = vis.solutionConfigurations(planner8, solution8)
vis.showMotion(collisionChecker8, path8, "Lazy-PRM, eight joints")

# Summary

* A robot with joints is planned in its **joint space**; the planners stay unchanged, only the collision checker is exchanged.
* The collision checker maps a configuration to the work space (forward kinematics) and checks the geometry of the robot there. The C-obstacles are never computed.
* A straight line in the joint space is a curved motion in the work space. It is checked by sampling: the **resolution** has to fit the robot and the scene, otherwise collisions are missed. Paths should be validated independently.
* The model has limits: no self-collisions, no joint limits in the collision check, joint angles are not treated as periodic, and the links are lines or capsules.
* Further notebooks with this robot: IP-10-1 (tests with 2, 3 and 4 joints), IP-11-0 (BB-Method with a robot with 10 joints), IP-12-0 (smoothing in the joint space).

# Experiments and Exercises

1. Change start and goal (e.g. use the two configurations of `inverseKinematics` for a TCP position as start and goal). When does the robot have to move "around" a C-obstacle?
2. Use a list with **both** inverse kinematics solutions as goal list of `planPath`. Which one do the planners use? How could a planner profit from several goals?
3. Find the largest `fk_resolution`, for which all paths of the experiment above are valid. How does it depend on the length of the links and on the thickness of the obstacles?
4. `lineInCollision` uses the same distance in the joint space for all joints. A rotation of the first joint moves the tip much further than a rotation of the last joint. Implement a line check, whose step size limits the motion of the tip in the work space.
5. Add the joint limits to the collision check (subclass of `KinChainCollisionChecker`, compare `CollisionChecker.isInLimits`) and plan with limits of $\pm 2$ rad.
6. Add a check for self-collisions: two links, which are not neighbours, must not intersect. For which number of joints does it matter?
7. Run A\* (IPAStar.py, IP-8-0) in the joint space of the two-link robot, e.g. with `{"w": 0.5, "heuristic": "euclidean", "discretization": 0.1, "maxIterations": 5000}`. Compare the path and the number of collision checks with the PRMs. What happens with more joints?